# Forecasting NSE Stock Prices with a Hybrid EGARCH–XGBoost Model

## 1. Introduction

### 1.1 Background
The Nairobi Securities Exchange (NSE) is Kenya's main equity market. Many NSE stocks trade thinly, jump suddenly and move through calm and stormy periods, which makes them hard to forecast with a single model.

This project combines two models:
* **EGARCH** captures **volatility**: how much a price is likely to move, including the tendency of bad news to raise volatility more than good news.
* **XGBoost** learns patterns from **technical features** such as recent returns, momentum, trading range and volume.
* The **hybrid** model gives XGBoost the EGARCH volatility forecast as extra inputs.

### 1.2 Objectives
1. Prepare and explore daily NSE price data for the selected stocks.
2. Build EGARCH, XGBoost and hybrid EGARCH–XGBoost models.
3. Compare them with simple baselines on data the models have never seen.
4. Deploy the hybrid model as a simple forecast web app.

### 1.3 Data source
* **Source:** daily equity trading data from the Nairobi Securities Exchange. *(Add the exact provider here.)*
* **Format:** Excel files (one per year) in `data/raw/`
* **Period:** January 2017 to June 2026, trading days only
* **Stocks:** set in `TICKERS` in the next cell

### 1.4 Time split
| Set | Period | Use |
|---|---|---|
| Training | 2017 – 2023 | Fit the models |
| Validation | 2024 – 2025 | Tune the settings |
| Test | Jan – Jun 2026 | Final evaluation on unseen data |

The data is split by date and never shuffled, so no model sees the future.

> **Run order:** use **Restart → Run All**. Every cell builds new objects from earlier ones, so re-running a cell is safe.

In [ ]:
import warnings, re, glob
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
from arch import arch_model

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (12, 5), "figure.dpi": 110})
RNG = 42

# ------------------------------ SETTINGS ------------------------------
DATA_DIR = next((p for p in [Path("data/raw"), Path("../data/raw")] if p.exists()), Path("data/raw"))
FIG_DIR = DATA_DIR.parent.parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Your stock codes exactly as in the CODE column. None = all stocks with >= 1,000 trading days.
TICKERS = None   # e.g. ["ABSA", "BRIT", "COOP", "EQTY", "KCB", "KQ", "KEGN", "KPLC", "....", "...."]

TRAIN_END  = pd.Timestamp("2023-12-31")
VAL_START, VAL_END   = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-12-31")
TEST_START, TEST_END = pd.Timestamp("2026-01-01"), pd.Timestamp("2026-06-30")
EXTREME_RET = 0.15       # |daily log return| above this = split / bonus issue / data error

def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")

print("Data folder:", DATA_DIR.resolve())

## 2. Data variables

### 2.1 Raw variables
| Variable | Type | Description |
|---|---|---|
| `YEAR` | integer | Calendar year |
| `DATE` | date | Trading date |
| `CODE` | text | Stock ticker, e.g. SCOM, KQ |
| `NAME` | text | Company name |
| `12m Low` / `12m High` | KES | Lowest / highest price over the past 12 months |
| `Day Low` / `Day High` | KES | Lowest / highest traded price that day |
| `Day Price` | KES | Closing price (**main variable**) |
| `Previous` | KES | Previous day's closing price |
| `Change` / `Change%` | KES / % | Change from the previous close |
| `Volume` | shares | Shares traded that day |

### 2.2 Derived variables
| Variable | Meaning |
|---|---|
| `ret` | Daily log return $r_t=\ln(P_t/P_{t-1})$ |
| `target` | Next day's return $r_{t+1}$, what the models predict |
| `ret_lag1..5` | Returns over the last 5 days |
| `ret_ma5`, `ret_ma21`, `mom_63` | Short-term average return and 3-month momentum |
| `vol_5`, `vol_21`, `vol_63` | Rolling volatility (std of returns) |
| `range_pct` | (Day High − Day Low) / Day Price |
| `px_ma20_gap` | Price vs its 20-day average |
| `pos_12m` | Position in the 12-month range (0 = at low, 1 = at high) |
| `rsi_14` | 14-day Relative Strength Index |
| `logvol_z` | Today's volume vs its usual level |
| `stale_5` | No-change days in the last week (thin trading) |
| `mkt_ret` | Average return of all selected stocks that day |
| `stock_id` | Number identifying the stock |
| **`eg_vol`** | **EGARCH forecast of tomorrow's volatility** (hybrid input) |
| **`eg_z`** | **Today's EGARCH standardised shock** (hybrid input) |
| **`vol_gap`** | **`eg_vol` / `vol_21`**: is volatility expected to rise or fall? (hybrid input) |

## 3. Data preparation
### 3.1 Load the files

In [ ]:
COLMAP = {"year": "year", "date": "date", "code": "code", "name": "name",
          "12mlow": "low_12m", "12mhigh": "high_12m", "daylow": "day_low", "dayhigh": "day_high",
          "dayprice": "price", "previous": "previous", "change": "change",
          "change%": "change_pct", "volume": "volume"}

def to_num(s):
    if s.dtype.kind in "if":
        return s.astype(float)
    s = s.astype(str).str.replace(",", "").str.replace("%", "").str.strip()
    return pd.to_numeric(s.replace({"-": None, "": None, "nan": None}), errors="coerce")

files = sorted(f for f in glob.glob(str(DATA_DIR / "*"))
               if f.lower().endswith((".xlsx", ".xls", ".csv")) and not Path(f).name.startswith("~$"))
assert files, f"No .xlsx/.csv files found in {DATA_DIR.resolve()}"

frames = []
for f in files:
    d = pd.read_csv(f) if f.lower().endswith(".csv") else pd.read_excel(f)
    d.columns = [COLMAP.get(re.sub(r"[^a-z0-9%]", "", str(c).lower()), str(c)) for c in d.columns]
    frames.append(d)
    print(f"{Path(f).name:35s} {len(d):>8,} rows")

raw = pd.concat(frames, ignore_index=True)
raw["date"] = pd.to_datetime(raw["date"], dayfirst=True, errors="coerce")
raw["code"] = raw["code"].astype(str).str.strip().str.upper()
raw["name"] = raw["name"].astype(str).str.strip()
for c in ["low_12m", "high_12m", "day_low", "day_high", "price", "previous", "change", "change_pct", "volume"]:
    if c in raw.columns:
        raw[c] = to_num(raw[c])
print("\nCombined:", raw.shape)
raw.head()

### 3.2 Data quality checks

In [ ]:
display(pd.DataFrame({"missing": raw.isna().sum(), "missing_%": raw.isna().mean() * 100}))
print("Unreadable dates        :", raw["date"].isna().sum())
print("Duplicate (date, code)  :", raw.duplicated(["date", "code"]).sum())
print("Zero / negative prices  :", (raw["price"] <= 0).sum())
print("Price outside day range :", ((raw["price"] < raw["day_low"] - 1e-9) | (raw["price"] > raw["day_high"] + 1e-9)).sum())
print("Date range              :", raw["date"].min().date(), "to", raw["date"].max().date())

### 3.3 Cleaning, stock selection and returns

In [ ]:
df = (raw.dropna(subset=["date", "price"]).query("price > 0")
         .sort_values(["code", "date"]).drop_duplicates(["date", "code"], keep="last"))
if TICKERS:
    missing = set(TICKERS) - set(df["code"])
    if missing:
        print("Not found in the data:", missing)
    df = df[df["code"].isin(TICKERS)]
else:
    days = df.groupby("code")["date"].nunique()
    df = df[df["code"].isin(days[days >= 1000].index)]
df = df.reset_index(drop=True)

df["ret"] = df.groupby("code")["price"].transform(lambda p: np.log(p).diff())
df["extreme"] = df["ret"].abs() > EXTREME_RET
df["year"] = df["date"].dt.year

TICKS = sorted(df["code"].unique())
NAMES = df.groupby("code")["name"].last().to_dict()
print(f"{len(TICKS)} stocks: {TICKS}")
print(f"{len(df):,} rows | extreme moves flagged: {df['extreme'].sum()}")
df.loc[df["extreme"], ["date", "code", "previous", "price", "ret"]].head(15)

## 4. Exploratory data analysis
### 4.1 Summary by stock

In [ ]:
clean = df[~df["extreme"]].dropna(subset=["ret"])
summary = pd.DataFrame({
    "name": df.groupby("code")["name"].last(),
    "first_date": df.groupby("code")["date"].min().dt.date,
    "last_date": df.groupby("code")["date"].max().dt.date,
    "trading_days": df.groupby("code")["date"].nunique(),
    "min_price": df.groupby("code")["price"].min(),
    "max_price": df.groupby("code")["price"].max(),
    "last_price": df.groupby("code")["price"].last(),
    "median_volume": df.groupby("code")["volume"].median(),
    "no_change_days_%": clean.groupby("code")["ret"].apply(lambda r: (r == 0).mean() * 100),
    "annual_vol_%": clean.groupby("code")["ret"].std() * np.sqrt(252) * 100,
})
summary

### 4.2 Average price per stock per year

In [ ]:
avg_year = df.pivot_table(index="year", columns="code", values="price", aggfunc="mean")
display(avg_year.round(2))

ax = avg_year.T.plot.bar(figsize=(16, 6), width=0.85, colormap="viridis")
ax.set_title("Average closing price per stock per year (KES)"); ax.set_xlabel(""); ax.set_ylabel("KES")
ax.legend(title="Year", ncol=5, fontsize=8); plt.xticks(rotation=0)
plt.tight_layout(); savefig("01_avg_price_by_year"); plt.show()

In [ ]:
ncols = 5; nrows = int(np.ceil(len(TICKS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.4 * nrows), squeeze=False)
for ax, t in zip(axes.flat, TICKS):
    s = avg_year[t].dropna()
    ax.bar(s.index, s.values, color="#2a9d8f")
    ax.plot(s.index, s.values, "o-", color="#264653", lw=1)
    ax.set_title(t, weight="bold"); ax.tick_params(axis="x", rotation=45, labelsize=8)
for ax in list(axes.flat)[len(TICKS):]:
    ax.set_visible(False)
fig.suptitle("Average yearly price by stock (KES, each on its own scale)", y=1.01)
plt.tight_layout(); savefig("02_avg_price_panels"); plt.show()

yoy = avg_year.pct_change() * 100
fig, ax = plt.subplots(figsize=(14, 0.5 * len(TICKS) + 2))
sns.heatmap(yoy.T, annot=True, fmt=".0f", cmap="RdYlGn", center=0, ax=ax, cbar_kws={"label": "% change"})
ax.set_title("Year-on-year change in average price (%)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); savefig("03_yoy_heatmap"); plt.show()

### 4.3 Price movement and yearly swings

In [ ]:
px = df.pivot(index="date", columns="code", values="price").sort_index()
rebased = px.apply(lambda s: s / s.dropna().iloc[0] * 100)
fig, ax = plt.subplots(figsize=(15, 6))
rebased.plot(ax=ax, lw=1.1, logy=True)
ax.axhline(100, color="k", lw=0.6, ls="--")
for d, lab in [(VAL_START, "Validation"), (TEST_START, "Test")]:
    ax.axvline(d, color="grey", ls=":"); ax.text(d, ax.get_ylim()[1] * 0.8, f" {lab}", color="grey")
ax.set_title("Prices rebased to 100 (log scale)"); ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); savefig("04_rebased"); plt.show()

lo = df.pivot_table(index="year", columns="code", values="price", aggfunc="min")
hi = df.pivot_table(index="year", columns="code", values="price", aggfunc="max")
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.2 * nrows), squeeze=False)
for i, (ax, t) in enumerate(zip(axes.flat, TICKS)):
    c = plt.cm.tab10(i % 10)
    ax.fill_between(lo.index, lo[t], hi[t], color=c, alpha=0.25)
    ax.plot(avg_year.index, avg_year[t], "o-", color=c, ms=3)
    ax.set_title(t, weight="bold"); ax.tick_params(labelsize=8)
for ax in list(axes.flat)[len(TICKS):]:
    ax.set_visible(False)
fig.suptitle("Yearly swing: lowest-to-highest price (band) and yearly average (line)", y=1.01)
plt.tight_layout(); savefig("05_yearly_swing"); plt.show()

### 4.4 Liquidity

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
summary["median_volume"].sort_values().plot.barh(ax=ax[0], color="#264653", logx=True)
ax[0].set_title("Median daily volume (shares, log scale)")
summary["no_change_days_%"].sort_values().plot.barh(ax=ax[1], color="#f4a261")
ax[1].set_title("Days with no price change (%)")
plt.tight_layout(); savefig("06_liquidity"); plt.show()

### 4.5 Return distributions

In [ ]:
dist = clean.groupby("code")["ret"].agg(
    mean_pct=lambda r: r.mean() * 100, std_pct=lambda r: r.std() * 100,
    skew=lambda r: stats.skew(r), excess_kurtosis=lambda r: stats.kurtosis(r),
    jarque_bera_p=lambda r: stats.jarque_bera(r).pvalue)
display(dist)

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.2 * nrows), squeeze=False)
for ax, t in zip(axes.flat, TICKS):
    r = clean.loc[clean["code"] == t, "ret"]
    x = np.linspace(r.quantile(0.001), r.quantile(0.999), 200)
    ax.hist(r, bins=100, density=True, color="#264653", alpha=0.6)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), color="#e76f51")
    ax.set_xlim(x.min(), x.max()); ax.set_title(t, weight="bold")
for ax in list(axes.flat)[len(TICKS):]:
    ax.set_visible(False)
fig.suptitle("Daily returns vs the normal curve (orange): tall peaks and fat tails", y=1.01)
plt.tight_layout(); savefig("07_return_distributions"); plt.show()

### 4.6 Volatility clustering, stationarity and ARCH effects

In [ ]:
rets = clean.pivot(index="date", columns="code", values="ret").sort_index()
roll_vol = rets.rolling(63, min_periods=40).std() * np.sqrt(252) * 100
fig, ax = plt.subplots(figsize=(15, 5))
roll_vol.plot(ax=ax, lw=1)
ax.set_title("3-month rolling volatility (% per year): calm and stormy periods cluster")
ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); savefig("08_rolling_vol"); plt.show()

top = summary["median_volume"].idxmax()
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(rets[top].dropna(), lags=30, ax=ax[0], title=f"{top}: autocorrelation of returns")
plot_acf(rets[top].dropna() ** 2, lags=30, ax=ax[1], title=f"{top}: autocorrelation of squared returns")
plt.tight_layout(); savefig("09_acf"); plt.show()

tests = pd.DataFrame({t: {
    "ADF_log_price_p": adfuller(np.log(px[t].dropna()))[1],
    "ADF_return_p": adfuller(rets[t].dropna())[1],
    "LjungBox_sq_ret_p": acorr_ljungbox(rets[t].dropna() ** 2, lags=[10])["lb_pvalue"].iloc[0],
    "ARCH_LM_p": het_arch(rets[t].dropna())[1]} for t in TICKS}).T
display(tests)
print(f"Returns stationary (ADF p<0.05): {(tests['ADF_return_p'] < 0.05).sum()}/{len(TICKS)}")
print(f"ARCH effects (p<0.05): {(tests['ARCH_LM_p'] < 0.05).sum()}/{len(TICKS)}")

### 4.7 Correlation between stocks

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(rets.loc[:TRAIN_END].corr(min_periods=250), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation of daily returns (training period)")
plt.tight_layout(); savefig("10_correlation"); plt.show()

### 4.8 EDA takeaways
* Price levels and trends differ a lot between stocks, so the models use **returns**, not prices.
* Returns have **fat tails**, so EGARCH uses Student-t errors.
* Volatility **clusters**, which supports a GARCH-type model.
* Many stocks have frequent no-change days (thin trading).

## 5. Modelling

### 5.1 How the features and EGARCH are built
Everything is built **per stock in one function**, and the stocks are then stacked into a single table called `panel`. Nothing is merged in later, which is what caused errors in the earlier version.

**EGARCH(1,1,1) with Student-t errors**, fitted on **2017–2023 only**:
$$\ln\sigma_{t+1}^2 = \omega + \alpha\big(|z_t| - \sqrt{2/\pi}\big) + \gamma z_t + \beta\ln\sigma_t^2$$

The fitted parameters are then **frozen** and applied to the full series (`.fix()`), so validation and test forecasts never use future data.
For each day *t*, `eg_vol` is the forecast of tomorrow's volatility $\sigma_{t+1}$. It uses only today's shock $z_t$ and today's $\sigma_t$, so it is known at today's close.

In [ ]:
def rsi(p, n=14):
    d = p.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + up / dn.replace(0, np.nan))

def egarch_features(r):
    # EGARCH fitted on training data only, then applied to the whole series with frozen parameters
    r_pct = (r.dropna() * 100).clip(-10, 10)
    train = r_pct[r_pct.index <= TRAIN_END]
    spec = dict(mean="Constant", vol="EGARCH", p=1, o=1, q=1, dist="t", rescale=False)
    fit = arch_model(train, **spec).fit(disp="off", options={"maxiter": 500})
    p = fit.params
    full = arch_model(r_pct, **spec).fix(p)
    sigma = full.conditional_volatility                     # sigma_t (in %)
    z = (r_pct - p["mu"]) / sigma                           # today's standardised shock
    log_var_next = (p["omega"] + p["alpha[1]"] * (z.abs() - np.sqrt(2 / np.pi))
                    + p["gamma[1]"] * z + p["beta[1]"] * np.log(sigma ** 2))
    out = pd.DataFrame({"eg_vol": np.sqrt(np.exp(log_var_next)) / 100,   # tomorrow's vol (fraction)
                        "eg_z": z,
                        "eg_mu": p["mu"] / 100})
    return out, p

mkt_ret = clean.groupby("date")["ret"].mean()
stock_frames, eg_params = [], {}

for i, t in enumerate(TICKS):
    s = df[df["code"] == t].set_index("date").sort_index().copy()
    r = s["ret"].where(~s["extreme"])                       # remove splits / bonus jumps
    f = pd.DataFrame(index=s.index)
    f["code"], f["name"], f["stock_id"], f["price"] = t, s["name"], i, s["price"]
    f["ret"] = r
    for k in range(1, 6):
        f[f"ret_lag{k}"] = r.shift(k - 1)                   # lag1 = today's return
    f["ret_ma5"] = r.rolling(5, min_periods=3).mean()
    f["ret_ma21"] = r.rolling(21, min_periods=10).mean()
    f["mom_63"] = r.rolling(63, min_periods=40).sum()
    for w in [5, 21, 63]:
        f[f"vol_{w}"] = r.rolling(w, min_periods=max(3, w // 2)).std()
    f["range_pct"] = (s["day_high"] - s["day_low"]) / s["price"]
    f["px_ma20_gap"] = s["price"] / s["price"].rolling(20).mean() - 1
    f["pos_12m"] = (s["price"] - s["low_12m"]) / (s["high_12m"] - s["low_12m"]).replace(0, np.nan)
    f["rsi_14"] = rsi(s["price"])
    lv = np.log1p(s["volume"].fillna(0))
    f["logvol_z"] = (lv - lv.rolling(63, min_periods=20).mean()) / lv.rolling(63, min_periods=20).std()
    f["stale_5"] = (r == 0).astype(float).rolling(5).sum()
    f["mkt_ret"] = mkt_ret.reindex(f.index)
    f["dow"] = f.index.dayofweek
    f["target"] = r.shift(-1)                               # tomorrow's return

    if (r.dropna().index <= TRAIN_END).sum() >= 500:
        try:
            eg, params = egarch_features(r)
            f = f.join(eg)
            eg_params[t] = params
        except Exception as e:
            print(f"EGARCH failed for {t}: {e}")
    else:
        print(f"{t}: fewer than 500 training days, no EGARCH")
    stock_frames.append(f)

panel = pd.concat(stock_frames).reset_index().replace([np.inf, -np.inf], np.nan)
for c in ["eg_vol", "eg_z", "eg_mu"]:
    if c not in panel.columns:
        panel[c] = np.nan
panel["vol_gap"] = panel["eg_vol"] / panel["vol_21"]
eg_params = pd.DataFrame(eg_params).T

BASE = ([f"ret_lag{k}" for k in range(1, 6)] + ["ret_ma5", "ret_ma21", "mom_63", "vol_5", "vol_21", "vol_63",
        "range_pct", "px_ma20_gap", "pos_12m", "rsi_14", "logvol_z", "stale_5", "mkt_ret", "dow", "stock_id"])
HYBRID = BASE + ["eg_vol", "eg_z", "vol_gap"]

missing = [c for c in HYBRID + ["target"] if c not in panel.columns]
assert not missing, f"Missing columns: {missing}"
print(f"panel: {panel.shape} | EGARCH fitted for {len(eg_params)}/{len(TICKS)} stocks")
print(f"Rows with an EGARCH volatility value: {panel['eg_vol'].notna().mean():.1%}")
panel[["date", "code", "price", "ret", "target", "vol_21", "eg_vol", "eg_z", "vol_gap"]].tail()

### 5.2 EGARCH parameters

In [ ]:
display(eg_params.round(4))
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
eg_params["beta[1]"].sort_values().plot.barh(ax=ax[0], color="#264653")
ax[0].set_title("β: volatility persistence (close to 1 = long-lasting)")
g = eg_params["gamma[1]"].sort_values()
g.plot.barh(ax=ax[1], color=np.where(g < 0, "#e76f51", "#2a9d8f"))
ax[1].axvline(0, color="k", lw=0.7); ax[1].set_title("γ: asymmetry (negative = leverage effect)")
eg_params["nu"].sort_values().plot.barh(ax=ax[2], color="#e9c46a")
ax[2].set_title("ν: tail thickness (lower = fatter tails)")
plt.tight_layout(); savefig("11_egarch_params"); plt.show()
print(f"Median β = {eg_params['beta[1]'].median():.3f} | stocks with γ < 0: {(eg_params['gamma[1]'] < 0).sum()}/{len(eg_params)}")

### 5.3 Train / validation / test split

In [ ]:
data = panel.dropna(subset=["target"]).copy()
data["split"] = np.select(
    [data["date"] <= TRAIN_END,
     data["date"].between(VAL_START, VAL_END),
     data["date"].between(TEST_START, TEST_END)],
    ["train", "val", "test"], default="out")
data = data[data["split"] != "out"].reset_index(drop=True)
data.groupby("split").agg(rows=("target", "size"), start=("date", "min"), end=("date", "max"),
                          stocks=("code", "nunique")).loc[["train", "val", "test"]]

### 5.4 EGARCH volatility check
How well does `eg_vol` track the actual size of tomorrow's move, compared with a simple 21-day rolling volatility?

In [ ]:
rows = []
for s in ["val", "test"]:
    d = data[data["split"] == s].dropna(subset=["eg_vol", "vol_21"])
    y = d["target"].abs()
    for name, col in [("EGARCH", "eg_vol"), ("Rolling 21-day", "vol_21")]:
        rows.append({"split": s, "volatility model": name,
                     "corr(forecast, |actual|)": np.corrcoef(d[col], y)[0, 1],
                     "MSE of variance (x1e6)": mean_squared_error(y ** 2, d[col] ** 2) * 1e6})
display(pd.DataFrame(rows).set_index(["split", "volatility model"]))

d = data[(data["code"] == top) & (data["split"] != "train")]
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(d["date"], d["target"].abs() * 100, color="grey", lw=0.6, label="|actual return| %")
ax.plot(d["date"], d["eg_vol"] * 100, color="#e76f51", lw=1.6, label="EGARCH volatility forecast %")
ax.axvline(TEST_START, color="k", ls=":"); ax.legend()
ax.set_title(f"{top}: out-of-sample EGARCH volatility forecast")
plt.tight_layout(); savefig("12_egarch_vol"); plt.show()

### 5.5 Baselines and evaluation function
* **Random walk:** predicts no change tomorrow
* **Historical mean:** each stock's average training return
* **EGARCH mean:** the EGARCH constant mean $\mu$

Metrics: RMSE and MAE of the next-day return (%), **directional accuracy** (days with a price change only) and **price MAPE**.

In [ ]:
def evaluate(d, col):
    m = d.dropna(subset=[col, "target"])
    y, p = m["target"], m[col]
    nz = y != 0
    return {"RMSE_%": np.sqrt(mean_squared_error(y, p)) * 100,
            "MAE_%": mean_absolute_error(y, p) * 100,
            "Direction_acc": (np.sign(p[nz]) == np.sign(y[nz])).mean() if (p != 0).any() else np.nan,
            "Price_MAPE_%": (np.abs(np.exp(p) - np.exp(y)) / np.exp(y)).mean() * 100}

train_mean = data[data["split"] == "train"].groupby("code")["target"].mean()
predictions = pd.DataFrame(index=data.index)
predictions["Random walk"] = 0.0
predictions["Historical mean"] = data["code"].map(train_mean).fillna(0)
predictions["EGARCH"] = data["eg_mu"]
predictions.head()

### 5.6 XGBoost and Hybrid EGARCH–XGBoost
The same function trains both models; only the feature list changes:
* **XGBoost:** `BASE` features (20)
* **Hybrid:** `BASE` + `eg_vol`, `eg_z`, `vol_gap` (23)

**Tuning:** a small grid (tree depth, learning rate, minimum leaf size), scored on validation RMSE, with early stopping.
**Final model:** refit on train + validation with the chosen settings, then scored once on test.

In [ ]:
def train_xgb(features, label):
    tr, va = data[data["split"] == "train"], data[data["split"] == "val"]
    common = dict(subsample=0.8, colsample_bytree=0.8, objective="reg:pseudohubererror",
                  huber_slope=0.02, tree_method="hist", random_state=RNG, n_jobs=-1)
    grid = []
    best = None
    for depth, lr, mcw in product([3, 5], [0.03, 0.1], [5, 50]):
        m = xgb.XGBRegressor(n_estimators=1500, max_depth=depth, learning_rate=lr,
                             min_child_weight=mcw, early_stopping_rounds=100, **common)
        m.fit(tr[features], tr["target"], eval_set=[(va[features], va["target"])], verbose=False)
        rmse = np.sqrt(mean_squared_error(va["target"], m.predict(va[features]))) * 100
        grid.append({"max_depth": depth, "learning_rate": lr, "min_child_weight": mcw,
                     "trees": m.best_iteration + 1, "val_RMSE_%": rmse})
        if best is None or rmse < best["rmse"]:
            best = {"rmse": rmse, "model": m, "depth": depth, "lr": lr, "mcw": mcw}

    final = xgb.XGBRegressor(n_estimators=max(best["model"].best_iteration + 1, 50),
                             max_depth=best["depth"], learning_rate=best["lr"],
                             min_child_weight=best["mcw"], **common)
    trva = data[data["split"].isin(["train", "val"])]
    final.fit(trva[features], trva["target"])

    pred = pd.Series(best["model"].predict(data[features]), index=data.index)
    is_test = data["split"] == "test"
    pred[is_test] = final.predict(data.loc[is_test, features])
    print(f"{label}: max_depth={best['depth']}, learning_rate={best['lr']}, "
          f"min_child_weight={best['mcw']}, trees={best['model'].best_iteration + 1}")
    return best["model"], final, pred, pd.DataFrame(grid).sort_values("val_RMSE_%")

xgb_tuned, xgb_final, predictions["XGBoost"], xgb_grid = train_xgb(BASE, "XGBoost")
hyb_tuned, hyb_final, predictions["Hybrid EGARCH-XGBoost"], hyb_grid = train_xgb(HYBRID, "Hybrid")
display(hyb_grid.head())

### 5.7 Model comparison

In [ ]:
results = pd.DataFrame([
    {"split": s, "model": m, **evaluate(data[data["split"] == s].assign(pred=predictions[m]), "pred")}
    for s in ["val", "test"] for m in predictions.columns]).set_index(["split", "model"])

display(results.style.format({"RMSE_%": "{:.4f}", "MAE_%": "{:.4f}", "Direction_acc": "{:.1%}",
                              "Price_MAPE_%": "{:.3f}"})
        .highlight_min(subset=["RMSE_%", "MAE_%", "Price_MAPE_%"], color="#d8f3dc")
        .highlight_max(subset=["Direction_acc"], color="#d8f3dc"))

plot = results.reset_index()
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
for a, m in zip(ax, ["RMSE_%", "MAE_%", "Direction_acc"]):
    sns.barplot(data=plot, x="model", y=m, hue="split", ax=a, palette=["#e9c46a", "#e76f51"])
    lo_, hi_ = plot[m].min(), plot[m].max()
    a.set_ylim(lo_ - (hi_ - lo_) * 0.6 - 1e-9, hi_ + (hi_ - lo_) * 0.3 + 1e-9)
    a.set_title(m); a.set_xlabel(""); a.tick_params(axis="x", rotation=30)
ax[2].axhline(0.5, color="k", ls="--", lw=0.8)
plt.tight_layout(); savefig("13_model_comparison"); plt.show()
results.to_csv(FIG_DIR.parent / "model_results.csv")

In [ ]:
# Diebold-Mariano test (squared-error loss, one step ahead, small-sample correction)
def dm_test(y, a, b):
    d = ((y - a) ** 2 - (y - b) ** 2).dropna()
    T = len(d)
    stat = d.mean() / np.sqrt(d.var(ddof=0) / T) * np.sqrt((T - 1) / T)
    return stat, 2 * stats.t.sf(abs(stat), T - 1)

test_idx = data.index[data["split"] == "test"]
y_test = data.loc[test_idx, "target"]
rows = []
for a, b in [("Hybrid EGARCH-XGBoost", "Random walk"), ("Hybrid EGARCH-XGBoost", "XGBoost"),
             ("Hybrid EGARCH-XGBoost", "EGARCH"), ("XGBoost", "Random walk")]:
    st_, p_ = dm_test(y_test, predictions.loc[test_idx, a], predictions.loc[test_idx, b])
    verdict = (f"{a} better" if st_ < 0 else f"{b} better") if p_ < 0.05 else "no significant difference"
    rows.append({"comparison": f"{a} vs {b}", "DM statistic": st_, "p-value": p_, "result": verdict})
pd.DataFrame(rows)

In [ ]:
# Error per stock on the test set
per_stock = pd.DataFrame({
    m: data.loc[test_idx].assign(p=predictions.loc[test_idx, m]).groupby("code")
         .apply(lambda d: np.sqrt(mean_squared_error(d["target"], d["p"])) * 100)
    for m in ["Random walk", "XGBoost", "Hybrid EGARCH-XGBoost"]})
per_stock["Hybrid vs random walk %"] = (per_stock["Hybrid EGARCH-XGBoost"] / per_stock["Random walk"] - 1) * 100
display(per_stock.round(4))

fig, ax = plt.subplots(figsize=(12, 5))
per_stock[["Random walk", "XGBoost", "Hybrid EGARCH-XGBoost"]].plot.bar(ax=ax, colormap="viridis")
ax.set_title("Test RMSE (%) by stock"); ax.set_xlabel(""); plt.xticks(rotation=0)
plt.tight_layout(); savefig("14_per_stock_rmse"); plt.show()

### 5.8 Feature importance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 7))
for a, (m, feats, title) in zip(ax, [(xgb_tuned, BASE, "XGBoost"), (hyb_tuned, HYBRID, "Hybrid")]):
    imp = pd.Series(m.feature_importances_, index=feats).sort_values()
    imp.plot.barh(ax=a, color=["#e76f51" if c in ["eg_vol", "eg_z", "vol_gap"] else "#264653" for c in imp.index])
    a.set_title(f"{title}: feature importance (EGARCH inputs in orange)")
plt.tight_layout(); savefig("15_feature_importance"); plt.show()

### 5.9 Actual vs forecast price on the test set

In [ ]:
d = data[(data["code"] == top) & (data["split"] == "test")]
p = predictions.loc[d.index, "Hybrid EGARCH-XGBoost"]
actual_next = d["price"] * np.exp(d["target"])
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(d["date"], actual_next, color="k", lw=1.5, label="Actual next-day price")
ax.plot(d["date"], d["price"] * np.exp(p), color="#e76f51", ls=":", lw=2, label="Hybrid forecast")
ax.fill_between(d["date"], d["price"] * np.exp(p - 1.28 * d["eg_vol"]), d["price"] * np.exp(p + 1.28 * d["eg_vol"]),
                color="#e76f51", alpha=0.15, label="80% range (EGARCH)")
ax.set_title(f"{NAMES[top]}: next-day forecasts, Jan–Jun 2026"); ax.set_ylabel("KES"); ax.legend()
plt.tight_layout(); savefig("16_test_forecast"); plt.show()
inside = ((actual_next >= d["price"] * np.exp(p - 1.28 * d["eg_vol"])) &
          (actual_next <= d["price"] * np.exp(p + 1.28 * d["eg_vol"]))).mean()
print(f"Share of test days inside the 80% range: {inside:.1%}")

## 6. Deployment

### 6.1 Set-up
The hybrid model is served through a **Streamlit web app** (`app.py`) packaged with **Docker** (`Dockerfile.app`).

```
data/raw (Excel)  →  app.py  →  EGARCH volatility + Hybrid XGBoost  →  web page (port 8501)
                        └──────────── Docker container ────────────┘
```

The user picks a **stock** and a **forecast period** (1 day, 1 week, 2 weeks, 1 month or 3 months), then clicks **Forecast**. The page shows:
* the last price, the forecast price and an 80% likely range
* a chart with the forecast as a dotted line
* a disclaimer

### 6.2 From a one-day model to longer forecasts
1. **Day 1:** the hybrid model's return, capped at ±2× tomorrow's volatility.
2. **Later days:** the day-1 signal halves each day and moves toward the stock's small long-run trend.
3. **80% range:** volatility starts at the EGARCH forecast and fades back to the stock's normal level over the last 12 months, kept between ¼× and 3× of that level.

These safeguards keep ranges realistic for volatile, thinly traded stocks. The cell below reproduces this logic using the models trained above.

In [ ]:
HORIZONS = {"1 day": 1, "1 week": 5, "2 weeks": 10, "1 month": 21, "3 months": 63}
latest = panel.sort_values("date").groupby("code").tail(1).set_index("code")
latest_pred = pd.Series(hyb_final.predict(latest[HYBRID]), index=latest.index)

def app_forecast(code, horizon, z=1.28):
    row = latest.loc[code]
    r = panel.loc[panel["code"] == code, "ret"].dropna()
    vol_long = r.tail(252).std()
    vol_now = float(np.clip(row["eg_vol"] if pd.notna(row["eg_vol"]) else vol_long, 0.25 * vol_long, 3 * vol_long))
    beta = float(np.clip(eg_params.loc[code, "beta[1]"], 0.5, 0.98)) if code in eg_params.index else 0.9
    k = np.arange(horizon)
    daily_vol = np.sqrt(vol_long ** 2 + (vol_now ** 2 - vol_long ** 2) * beta ** k)
    next_ret = float(np.clip(latest_pred[code], -2 * vol_now, 2 * vol_now))
    drift = float(np.clip(r.tail(252).mean(), -0.002, 0.002))
    cum = np.cumsum(drift + (next_ret - drift) * 0.5 ** k)
    sd = np.sqrt(np.cumsum(daily_vol ** 2))
    last_px = row["price"]
    dates = pd.bdate_range(row["date"] + pd.Timedelta(days=1), periods=horizon)
    return pd.DataFrame({"date": dates, "forecast": last_px * np.exp(cum),
                         "low": last_px * np.exp(cum - z * sd), "high": last_px * np.exp(cum + z * sd)}), last_px

rows = []
for t in TICKS:
    for lab, h in HORIZONS.items():
        f, last_px = app_forecast(t, h)
        rows.append({"stock": t, "horizon": lab, "last_price": last_px, "forecast": f["forecast"].iloc[-1],
                     "low": f["low"].iloc[-1], "high": f["high"].iloc[-1]})
app_table = pd.DataFrame(rows)
app_table["change_%"] = (app_table["forecast"] / app_table["last_price"] - 1) * 100
print("Forecast change (%) by stock and horizon")
display(app_table.pivot(index="stock", columns="horizon", values="change_%")[list(HORIZONS)].round(2))
print("80% range width (% of last price)")
display(((app_table["high"] - app_table["low"]) / app_table["last_price"] * 100)
        .to_frame("w").assign(stock=app_table["stock"], horizon=app_table["horizon"])
        .pivot(index="stock", columns="horizon", values="w")[list(HORIZONS)].round(1))

In [ ]:
# Preview of the app chart: one stock, 1 month and 3 months
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
hist = df[df["code"] == top].set_index("date")["price"].tail(90)
for ax, lab in zip(axes, ["1 month", "3 months"]):
    f, last_px = app_forecast(top, HORIZONS[lab])
    x = [hist.index[-1]] + list(f["date"])
    ax.plot(hist.index, hist.values, color="#264653", lw=2, label="Actual price")
    ax.plot(x, [last_px] + list(f["forecast"]), color="#e76f51", ls=":", lw=2.5, label="Forecast")
    ax.fill_between(x, [last_px] + list(f["low"]), [last_px] + list(f["high"]),
                    color="#e76f51", alpha=0.15, label="Likely range (80%)")
    ax.set_title(f"{NAMES[top]}: {lab} forecast"); ax.set_ylabel("KES"); ax.legend()
plt.tight_layout(); savefig("17_app_preview"); plt.show()

### 6.3 Running the app
```bash
# Locally
pip install -r requirements-app.txt
streamlit run app.py                         # opens http://localhost:8501

# In Docker
docker build -f Dockerfile.app -t nse-app .
docker run --rm -p 8501:8501 nse-app         # then open http://localhost:8501
```

### 6.4 Limitations and next steps
* EGARCH parameters are fixed after 2023. Re-estimating them regularly could help.
* Daily returns are close to unpredictable, so the hybrid's advantage is mainly **one day ahead**. Longer forecasts rely mostly on volatility and the long-run trend.
* Thin trading creates short-term patterns that may not be tradable after costs.
* Possible next steps: automatic daily data refresh, quantile models for ranges, SHAP explanations and cloud hosting.

### 6.5 Disclaimer
These forecasts are for academic research only and are **not financial advice**.